## Context

This notebook accompanies User Guide section 1 (Prerequisites).  It verifies that the runtime, GPU, and core libraries are reachable, then loads the prebuilt data-science-practices fixture so every subsequent tutorial can chain off it without rebuilding.  Reading the stats panel at the end tells you whether the foundation is in place to follow notebooks 02 through 27.

In [ ]:
# Phase 1B preamble: bootstrap runtime + load shared helpers
# Bootstrap: portable repo-root locator — walks up from CWD looking for
# tutorials/user_guide/notebooks/_shared.py. Works in both the source
# clone (knowlytix/GMS) and the public mirror clone (knowlytix/gmsh).
import sys
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / 'tutorials' / 'user_guide' / 'notebooks' / '_shared.py').exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from tutorials.user_guide.notebooks import _helpers_phase1 as h1
h1.bootstrap_runtime()  # stubs modeva if absent
from tutorials.user_guide.notebooks import _shared as ug
import torch
DEVICE = ug.DEVICE
print(f'Device: {DEVICE}, CUDA available: {torch.cuda.is_available()}')


### Concept: what must be true before anything else runs

The governed-agent stack assumes four things.  First, Python is at least 3.12 because the codebase uses structural `match`, `frozen` dataclasses with slots, and the `X | Y` union syntax in runtime contexts.  Second, PyTorch is installed with a CUDA-enabled build, because every GMS primitive evaluates on the unit sphere $S^{m-1}$ and the embedding tensors live on the GPU.  Third, the three first-party packages `gms`, `docgms`, and `governed_agent` import cleanly from the repo root.  Fourth, a built knowledge store sits on disk: the rest of the tutorials assume the store is loadable, not buildable, so we do not retrain on every notebook run.

If any of those four checks fails, stop and fix the environment before continuing.  A failure here is an infrastructure problem, not an agent problem; debugging later notebooks while torch is missing wastes hours.

The `bootstrap_runtime()` call in the preamble is the one deliberate workaround in these notebooks: the default Jupyter kernel (`/usr/bin/python3`) does not see the cluster venv where `torch` lives, so we prepend the venv site-packages to `sys.path` before importing anything GPU-dependent.  Activated shells do not need this; the kernel does.

In [ ]:
import sys
import platform
print('Python:        ', platform.python_version())
print('Executable:    ', sys.executable)
print('Torch version: ', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:   ', torch.cuda.get_device_name(0))
    print('CUDA version:  ', torch.version.cuda)


In [ ]:
# First-party package versions (best effort)
import importlib
for pkg in ['knowlytix.core', 'knowlytix.knowledge', 'knowlytix.harness.governance', 'knowlytix.harness']:
    try:
        m = importlib.import_module(pkg)
        v = getattr(m, '__version__', 'n/a')
        print(f'{pkg:35s} {v}')
    except Exception as e:
        print(f'{pkg:35s} IMPORT FAILED: {e}')


In [ ]:
# Load the prebuilt fixture and read its stats
store = ug.load_ds_practices_store()
stats = store.stats()
for k, v in stats.items():
    print(f'  {k:14s} {v}')


In [ ]:
# Visualize fixture stats as a single bar chart
import matplotlib.pyplot as plt

labels = ['triples', 'entities', 'relations', 'enm_entries']
values = [stats['triples'], stats['entities'],
          stats['relations'], stats['enm_entries']]
fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.bar(labels, values, color='tab:blue')
for x, v in zip(labels, values):
    ax.text(x, v + max(values) * 0.01, str(v),
            ha='center', va='bottom', fontsize=9)
ax.set_ylabel('count')
ax.set_title('ds_practices_store contents')
ax.set_ylim(0, max(values) * 1.15)
fig.tight_layout()
plt.show()


### Exercises

1. Run `nvidia-smi` from a shell and compare the GPU name there with what `torch.cuda.get_device_name(0)` reports above.  They should agree byte for byte.
2. Print `store.markdown[:600]` and confirm the source is the data-science-practices document that the manifest lists in `fixtures/MANIFEST.json`.
3. Re-run this notebook and confirm the stats are identical across runs.  The fixture is frozen on disk, so a stat drift means something corrupted the store.

### What you should now be able to do

- Verify the four prerequisites in one cell each.
- Load the shared fixture in a single call.
- Read the stats schema (entities, relations, triples, enm_entries) returned by `GMSExpertStore.stats()`.
- Recognise an environment failure (import error, no CUDA, missing fixture) before it surfaces in a later notebook.

### Cross-references

- USER_GUIDE.md section 1 (Prerequisites)
- `gms_agent_builder_ext/chapters/ch01.tex` for the   full architectural prelude
- `docgms/store.py:GMSExpertStore.stats` (source of   the stats keys used above)

In [ ]:
# Final assertion: the fixture is populated.
assert stats['triples'] >= 100, \
    f"fixture seems empty: {stats}"
print(f"OK: fixture has {stats['triples']} triples, "
      f"{stats['enm_entries']} ENM entries.")
